In [3]:
import pandas as pd
import json
import os
import glob

def compile_variations_log(base_dir='synthetic'):
    log_files = glob.glob(os.path.join(base_dir, 'variation_*', 'config_log.json'))
    records = []
    
    for fpath in log_files:
        with open(fpath, 'r') as f:
            data = json.load(f)

        data['variation_id'] = os.path.basename(os.path.dirname(fpath))
        records.append(data)
        
    if not records:
        print("Nenhuma variação encontrada.")
        return pd.DataFrame()
        
    df   = pd.DataFrame(records)
    cols = ['variation_id', 'iou'] + [c for c in df.columns if c not in ['variation_id', 'iou']]
    df   = df[cols]
    
    df = df.sort_values(by='iou', ascending=False).reset_index(drop=True)
    return df


df_variations = compile_variations_log()
df_variations

,variation_id,iou,shape,margin,layerRange,layerThickness,foldCount,foldSigma,foldAmplitude,foldDamping,...,faultRoughSigma,faultDecaySigma,faultZoneWidth,faultThreshold,faultCurveProb,faultCurveMax,waveletFreq,waveletDuration,waveletDt,noiseLevel
0,variation_462,0.765676,"[256, 256, 256]",64,"[89, 228]","[2, 5]","[6, 29]","[16, 31]","[-6, 17]",1.473985,...,3.251009,"[17, 97]",1.034943,0.855810,0.195110,3.937691,"[90, 136]",0.085993,0.008779,"[0, 0.1849609129870732]"
1,variation_316,0.761365,"[256, 256, 256]",64,"[80, 179]","[2, 5]","[6, 21]","[12, 37]","[-7, 16]",0.926920,...,6.760992,"[44, 88]",1.110002,0.609952,0.242989,3.946777,"[71, 110]",0.062412,0.008764,"[0, 0.19648082166093128]"
2,variation_154,0.754352,"[256, 256, 256]",64,"[59, 293]","[2, 3]","[7, 29]","[17, 38]","[-8, 14]",1.965289,...,3.001437,"[34, 83]",1.021975,0.699233,0.245811,3.149240,"[69, 97]",0.066042,0.009709,"[0, 0.15563172083662333]"
3,variation_258,0.753373,"[256, 256, 256]",64,"[97, 254]","[1, 5]","[8, 24]","[13, 37]","[-7, 18]",1.395223,...,4.268276,"[31, 61]",1.139553,0.500598,0.176761,3.639570,"[46, 147]",0.118693,0.006997,"[0, 0.16169499636921134]"
4,variation_197,0.746651,"[256, 256, 256]",64,"[72, 239]","[2, 4]","[9, 21]","[10, 26]","[-10, 10]",1.131906,...,2.995764,"[29, 86]",1.129978,0.850802,0.181543,3.109127,"[78, 113]",0.060444,0.002248,"[0, 0.160999392359087]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
512,variation_4,0.179753,"[256, 256, 256]",64,"[60, 288]","[2, 3]","[9, 37]","[22, 31]","[-13, 17]",1.106418,...,2.561681,"[16, 93]",0.562819,0.640357,0.355881,4.889356,"[40, 104]",0.058390,0.002881,"[0, 0.1]"
513,variation_139,0.167212,"[256, 256, 256]",64,"[96, 234]","[2, 4]","[8, 29]","[13, 43]","[-7, 19]",1.340998,...,5.749580,"[45, 66]",1.255251,0.535602,0.143398,3.372408,"[47, 124]",0.055654,0.000132,"[0, 0.19155179647670179]"
514,variation_41,0.146446,"[256, 256, 256]",64,"[106, 197]","[1, 4]","[17, 37]","[23, 43]","[-11, 11]",0.654701,...,4.740956,"[48, 81]",0.532139,0.706284,0.287072,5.202200,"[47, 148]",0.053367,0.004999,"[0, 0.06159189543124397]"
515,variation_496,0.111655,"[256, 256, 256]",64,"[96, 170]","[2, 4]","[5, 24]","[10, 32]","[-7, 11]",1.586191,...,5.622312,"[19, 70]",1.326507,0.585351,0.173838,3.129319,"[47, 117]",0.059524,0.000114,"[0, 0.18306243679817807]"


In [2]:
import ast
import warnings
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from scipy.optimize import differential_evolution
from sklearn.ensemble import RandomForestRegressor


class MultivariateIoUOptimizer:
    def __init__(self, dataframe: pd.DataFrame, target_col: str = 'iou', drop_cols: list = None):
        if drop_cols is None:
            drop_cols = ['variation_id']
        self.df = dataframe.copy()
        self.target_col = target_col
        self.drop_cols = drop_cols
        self.model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
        self.features = []
        self.X = None
        self.y = None

    def _parse_list_columns(self):
        original_cols = list(self.df.columns)
        for col in original_cols:
            if col == self.target_col or col in self.drop_cols:
                continue
            
            if self.df[col].dropna().empty:
                continue
                
            first_valid = self.df[col].dropna().iloc[0]
            
            if isinstance(first_valid, str) and first_valid.strip().startswith('[') and first_valid.strip().endswith(']'):
                self.df[col] = self.df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
                first_valid = self.df[col].dropna().iloc[0]
            
            if isinstance(first_valid, list):
                self.df[f'{col}_min'] = self.df[col].apply(lambda x: np.min(x) if isinstance(x, list) and len(x) > 0 else np.nan)
                self.df[f'{col}_max'] = self.df[col].apply(lambda x: np.max(x) if isinstance(x, list) and len(x) > 0 else np.nan)
                self.df[f'{col}_mean'] = self.df[col].apply(lambda x: np.mean(x) if isinstance(x, list) and len(x) > 0 else np.nan)
                self.df.drop(columns=[col], inplace=True)

    def _remove_zero_variance(self):
        numeric_df = self.df.select_dtypes(include=[np.number])
        variances = numeric_df.var()
        cols_to_drop = variances[variances == 0].index
        self.df.drop(columns=cols_to_drop, inplace=True)

    def preprocess(self):
        self._parse_list_columns()
        self._remove_zero_variance()
        
        self.df.drop(columns=[c for c in self.drop_cols if c in self.df.columns], inplace=True, errors='ignore')
        self.df.dropna(inplace=True)
        
        self.y = self.df[self.target_col]
        self.X = self.df.drop(columns=[self.target_col])
        self.features = self.X.columns.tolist()

    def fit_model(self):
        if self.X is None or self.y is None:
            self.preprocess()
        self.model.fit(self.X, self.y)

    def analyze_impact_and_optima(self) -> pd.DataFrame:
        bounds = [(self.X[col].min(), self.X[col].max()) for col in self.features]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            def objective(x):
                return -self.model.predict(x.reshape(1, -1))[0]
            
            res = differential_evolution(objective, bounds, seed=42)
            
        global_optimum_x = res.x
        global_optimum_iou = -res.fun

        results = []
        for i, col in enumerate(self.features):
            corr, _ = spearmanr(self.X[col], self.y)
            direction = "Aumentar melhora" if corr > 0 else "Diminuir melhora"
            if abs(corr) < 0.1:
                direction = "Neutro/Não-linear"

            results.append({
                'Feature': col,
                'Spearman_Corr': round(corr, 4),
                'Impacto_Direcional': direction,
                'Valor_Min_Historico': round(bounds[i][0], 4),
                'Valor_Max_Historico': round(bounds[i][1], 4),
                'Valor_Otimo_Estimado': round(global_optimum_x[i], 4),
                'IoU_Esperado_No_Otimo': round(global_optimum_iou, 4),
                'Feature_Importance_RF': round(self.model.feature_importances_[i], 4)
            })

        return pd.DataFrame(results).sort_values(by='Feature_Importance_RF', ascending=False).reset_index(drop=True)


optimizer = MultivariateIoUOptimizer(dataframe=df_variations)
optimizer.preprocess()
optimizer.fit_model()

df_analysis = optimizer.analyze_impact_and_optima()
df_analysis

,Feature,Spearman_Corr,Impacto_Direcional,Valor_Min_Historico,Valor_Max_Historico,Valor_Otimo_Estimado,IoU_Esperado_No_Otimo,Feature_Importance_RF
0,faultDipAngle_mean,0.6435,Aumentar melhora,30.0000,67.5000,65.8411,0.7133,0.6199
1,waveletDt,0.0901,Neutro/Não-linear,0.0001,0.0649,0.0208,0.7133,0.1670
2,faultZoneWidth,-0.1811,Diminuir melhora,0.5232,1.9859,1.0220,0.7133,0.0587
3,faultDipAngle_min,0.5904,Aumentar melhora,10.0000,50.0000,48.5413,0.7133,0.0090
4,faultCurveMax,-0.4566,Diminuir melhora,3.0024,9.9074,4.0214,0.7133,0.0084
5,foldCount_min,-0.3420,Diminuir melhora,5.0000,20.0000,6.0979,0.7133,0.0077
6,noiseLevel_mean,0.3822,Aumentar melhora,0.0000,0.1807,0.0945,0.7133,0.0065
7,waveletFreq_max,0.0195,Neutro/Não-linear,91.0000,150.0000,124.0268,0.7133,0.0056
8,noiseLevel_max,0.3822,Aumentar melhora,0.0001,0.3615,0.1762,0.7133,0.0054
9,shearGradient_mean,-0.0028,Neutro/Não-linear,-0.0716,0.0733,-0.0275,0.7133,0.0048
